### ENV Setup
---

In [3]:
from agents import Agent, WebSearchTool, ModelSettings, Runner, trace, function_tool
from openai.types.responses.tool import WebSearchToolFilters
from dotenv import load_dotenv
from IPython.display import display, Markdown
from pydantic import Field, BaseModel
import asyncio
from weasyprint import HTML

In [10]:
load_dotenv(override=True)

True

### Car Parts Search Agent
---

In [8]:
SEARCH_INSTRUCTIONS = "You are a research assistan. Given a search term, you search the \
    web for that term and produce a consice summary of the results. The summary must \
    be 2-3 paragraphs long and less than 300 words. Capture the main points. Write clearly, no \
    need to have complete sentences or good grammar. This will be consumed by someone \
    synthesizing a report, so it's vital you capture the essence and ingore any fluff. \
    Do not include any additional commentary othen than the summary itself."  

In [9]:
search_agent = Agent(
    name="Search Agent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[
        WebSearchTool(
            user_location={
                "type": "approximate",
                "country": "LV",
                "city": "Riga",
                "region": "Riga",
            }, 
            filters=WebSearchToolFilters(                
                allowed_domains=[
                    "intercars.lv",
                    "trodo.lv",
                    "eparts.lv",
                ]
            ),
            search_context_size="low"
        )
    ],    
    model="gpt-5.5",
    model_settings=ModelSettings(tool_choice="required")
)

In [38]:
search_prompt="budget friendly E92 brake kit"

with trace("Search"):
    result = await Runner.run(search_agent, search_prompt)

display(Markdown(result.final_output))

Trodo.lv shows several budget-friendly BMW 3 Coupe E92 brake parts, mainly brake pads rather than full “big brake kit”. Cheapest rear pad options: DAVID VASCO Z1612 at 17.22 €, TOPRAN 501 249 at 17.27 €, RAP BRAKES R-P0675 at 18.99 €. Front budget pad example: ABE C1B026ABE at 28.33 €, listed for BMW 3 E90/E91/E92/E93 and X1, ATE brake system, no wear sensor. ([trodo.lv](https://www.trodo.lv/bremzu-uzliku-komplekts-david-vasco-z1612?utm_source=openai))

Also useful low-cost supporting parts: MAXGEAR 27-0727 disc brake pad fitting kit around 2.52 €, QUICK BRAKE fitting kit around 2.63 €, BOSCH fitting kit around 5.67 €, and MAXGEAR brake hose 52-0342 around 5.44 €. Trodo listings often mark these as “Budžeta klase” and compatible with BMW 3 Coupe E92, but exact fit depends on engine/brake size. ([trodo.lv](https://www.trodo.lv/disku-bremzu-piederumi/bmw-3-coupe-e92-328-i-172kw-93582-cid?utm_source=openai))

Best budget route: avoid performance “big brake” kits; buy OE-size discs + pads + wear sensors/fitting kits matched by VIN or exact E92 engine. Search results from allowed sites did not clearly show a complete E92 brake kit bundle from intercars.lv, trodo.lv, or eparts.lv, only individual components.

### Part Finding Planner Agent
---

In [14]:
SEARCH_AMOUNT = 5

In [15]:
PLANNER_INSTRUCTIONS = f"""
You are an automotive parts search-query planner.

Given a user's request, generate exactly {SEARCH_AMOUNT} highly specific,
short search queries for automotive parts stores and catalogs.

Your queries must be optimized for parts-catalog search, not general web search.

RULES:
- Use precise automotive terminology and catalog-style keywords.
- Include vehicle make, model, chassis/generation (e.g. E92), and year/engine
  when provided and relevant.
- Include the exact part type requested (e.g. brake discs, brake pads, clutch kit).
- Preserve important specifications such as ceramic, drilled, vented, slotted,
  performance, OEM, front, rear, etc.
- Prefer compact keyword combinations over natural-language sentences.
- Do NOT write questions or conversational queries.
- Do NOT use filler such as "best", "please", "I need", "looking for",
  "recommend", "budget friendly", or "what should I buy".
- Do NOT make queries verbose.
- Do NOT invent vehicle specifications or part specifications that are not
  present in the user's request.
- Generate distinct search variants that could find different relevant products.
- When appropriate, vary one important catalog term at a time
  (e.g. "brake discs", "ceramic brake discs", "front brake discs pads").
- Use terminology commonly found in automotive parts catalogs.

Example:

User request:
"budget friendly E92 brake kit"

Good queries:
BMW E92 brake kit
BMW E92 brake discs pads
BMW E92 ceramic brake discs
BMW E92 front brake discs
BMW E92 sport brake kit

Bad queries:
best budget-friendly brake kit for BMW E92
what is the best brake kit for an E92
affordable high-performance ceramic brake system for BMW E92

Output ONLY the {SEARCH_AMOUNT} search queries, one query per line.
"""

In [16]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to answer the query.")

In [17]:
planner_agent = Agent(
    name="Planner Agent",
    instructions=PLANNER_INSTRUCTIONS,
    model="gpt-5.5",
    output_type=WebSearchPlan,
)

In [63]:
planner_prompt = "budget friendly E92 brake kit"

with trace("Car parts search"):
    result = await Runner.run(planner_agent, planner_prompt)
    print(result.final_output)

searches=[WebSearchItem(reason='Core catalog query with make, chassis, and requested part type.', query='BMW E92 brake kit'), WebSearchItem(reason='Catalog variant including common bundled components for brake kits.', query='BMW E92 brake discs pads'), WebSearchItem(reason='Variant preserving a common brake kit material/spec often used in catalogs.', query='BMW E92 ceramic brake kit'), WebSearchItem(reason='Position-specific variant for front axle brake kits.', query='BMW E92 front brake kit'), WebSearchItem(reason='Catalog synonym variant for performance-oriented brake kits without filler terms.', query='BMW E92 sport brake kit')]


### Report Agent
---

In [18]:
REPORT_INSTRUCTIONS = """
You are a senior automotive research analyst responsible for producing a
professional, decision-oriented report from a user's research query and
research gathered by other agents.

You will receive:
1. The original user query.
2. Research findings from automotive parts stores and other sources.

Your task is to transform the research into a detailed, cohesive HTML report
that can be converted directly into a PDF using WeasyPrint.

REPORT PROCESS:
1. Analyze the original query and identify the user's actual requirements.
2. Organize the available research into a logical report structure.
3. Compare relevant products and clearly distinguish meaningful differences.
4. Identify the best options based on the user's requirements.
5. Highlight important fitment, specifications, prices, availability, and
   other relevant information found in the research.
6. Clearly identify uncertainty, missing information, or specifications that
   could not be verified.
7. Do not invent product specifications, prices, compatibility, brands,
   availability, or other facts that are not present in the research.
8. Prefer concrete facts and comparisons over generic automotive advice.

REPORT STRUCTURE:
Use an appropriate structure for the particular query. When relevant,
include:

- Executive summary
- User requirements
- Vehicle / fitment information
- Search methodology or sources
- Product comparison
- Detailed product analysis
- Price comparison
- Technical specification comparison
- Advantages and disadvantages
- Best options / recommendations
- Important fitment considerations
- Final recommendation
- Sources

Do not force sections that are not relevant to the query.

PRODUCT COMPARISONS:
When comparing automotive parts, prioritize:

- Vehicle make/model/chassis
- Engine and year when relevant
- Front/rear position
- Part type
- Brand and manufacturer
- Part number
- Dimensions
- Material
- Construction
- Performance characteristics
- Certification or standards when available
- Price
- Availability
- Seller
- Product URL
- Fitment confidence

Use tables where they make comparisons easier to understand.

WRITING STYLE:
- Write like a professional automotive research analyst.
- Be factual, precise, and concise.
- Avoid marketing language and unsupported claims.
- Explain technical differences in practical terms.
- Do not repeat the same information unnecessarily.
- Use headings, subheadings, tables, bullet lists, and short paragraphs.
- Make the report useful for someone deciding which product to purchase.
- Target approximately 1,000-2,500 words depending on the amount of available
  research. Do not add filler just to reach a word count.

HTML REQUIREMENTS:
The final output must be a COMPLETE, VALID HTML DOCUMENT.

The HTML must:
- Start with <!DOCTYPE html>.
- Contain <html>, <head>, and <body>.
- Include all required CSS inside a <style> element.
- Be suitable for direct conversion to PDF using WeasyPrint.
- Use A4 page formatting.
- Use print-friendly colors and spacing.
- Use tables for structured product comparisons.
- Use page-break rules where appropriate.
- Avoid JavaScript.
- Do not depend on a hosted webpage or external CSS.
- Do not use Markdown.
- Do not wrap the HTML in Markdown code fences.
- Do not add commentary before or after the HTML.

PDF DESIGN:
Create a clean, professional automotive report.

Use:
- A4 page size
- Approximately 18mm page margins
- Clear typography
- Professional heading hierarchy
- Subtle borders and background colors
- Well-formatted comparison tables
- Highlight boxes for important findings
- Consistent spacing
- Page numbers where practical
- URLs as clickable links
- Avoid excessively large headings or wasted whitespace

If product images are provided in the research, you may include them using
their provided URLs. Do not invent image URLs.

Use CSS suitable for WeasyPrint, for example:

@page {
    size: A4;
    margin: 18mm;
}

Avoid CSS features that require JavaScript or browser-specific rendering.

SOURCE HANDLING:
Every important product claim should be traceable to the provided research.
Where source URLs are available, include them as clickable links in the report.

Do not fabricate citations or URLs.

FINAL OUTPUT:
Return ONLY the complete HTML document.

The HTML should follow this general structure, adapting it to the actual
research:

<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Automotive Parts Research Report</title>
<style>
@page {
    size: A4;
    margin: 18mm;
}

body {
    font-family: Arial, Helvetica, sans-serif;
    color: #222;
    font-size: 10.5pt;
    line-height: 1.5;
}

h1 {
    font-size: 24pt;
    color: #111;
    margin-bottom: 8px;
}

h2 {
    font-size: 17pt;
    color: #222;
    margin-top: 24px;
    page-break-after: avoid;
}

h3 {
    font-size: 13pt;
    margin-top: 18px;
    page-break-after: avoid;
}

p {
    margin: 6px 0 10px;
}

table {
    width: 100%;
    border-collapse: collapse;
    margin: 12px 0 18px;
    font-size: 9pt;
}

th {
    background: #222;
    color: white;
    font-weight: bold;
    text-align: left;
}

th, td {
    padding: 7px;
    border: 1px solid #ccc;
    vertical-align: top;
}

tr {
    page-break-inside: avoid;
}

.product {
    border: 1px solid #ddd;
    padding: 12px;
    margin: 12px 0;
    page-break-inside: avoid;
}

.price {
    font-size: 16pt;
    font-weight: bold;
    color: #111;
}

.highlight {
    background: #f3f6f8;
    border-left: 4px solid #333;
    padding: 10px 14px;
    margin: 12px 0;
    page-break-inside: avoid;
}

.warning {
    background: #fff4e5;
    border-left: 4px solid #e67e22;
    padding: 10px 14px;
    margin: 12px 0;
    page-break-inside: avoid;
}

a {
    color: #1558a6;
    text-decoration: none;
}

.footer {
    color: #777;
    font-size: 8pt;
    margin-top: 25px;
}
</style>
</head>

<body>

<!-- Generate the actual report here -->

</body>
</html>
"""

In [19]:
class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report stored using HTML markup")
    follow_up_questions: str = Field(description="Suggested topics to research further.")

writer_agent = Agent(
    name="Writer Agent",
    instructions=REPORT_INSTRUCTIONS,
    model="gpt-5.5",
    output_type=ReportData,
)

### PDF Generation Agent
---

In [29]:
async def generate_pdf(html: str):
    print("Generating PDF...")
    output_path = "report.pdf"
    HTML(string=html).write_pdf(output_path)
    print("PDF generated")
    return output_path

### Combining Search Planner and Search Agents
---

In [22]:
async def plan_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    search_prompt = f"Search term: {item.query}, \n Reason for searching: {item.reason}"
    result = await Runner.run(search_agent, search_prompt)
    return result.final_output

In [23]:
async def write_report(query: str, search_results: list[str]):
    print("Generating report...")
    input = f"Original query: {query}, \n Summarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

In [30]:
query = "budget friendly E92 brake kit"

with trace("Automted part research"):
    print("Starting reserach...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    pdf_file = await generate_pdf(report.markdown_report)
    print("Research finished!")

Starting reserach...
Planning searches...
Will perform 5 searches
Searching...
Finished searching
Generating report...
Finished writing report
Generating PDF...
PDF generated
Research finished!
